# TID Detection in Propagation Data

This notebook demonstrates techniques for detecting Travelling Ionospheric Disturbances (TIDs) using IONIS datasets.

**What you'll learn:**
- Understanding TIDs and their signature in propagation data
- Using SNR variance to detect anomalies
- Establishing baselines for comparison
- Analyzing specific events (rocket launches, geomagnetic storms, eclipses)

**Target audience:** HamSCI researchers, ionospheric scientists, PhD students studying TIDs.

## 1. Background: What are TIDs?

**Travelling Ionospheric Disturbances (TIDs)** are wavelike disturbances in the ionosphere caused by:

- Gravity waves from tropospheric sources
- Geomagnetic storms (auroral heating)
- Rocket launches (supersonic exhaust)
- Solar eclipses (thermal gradient)
- Earthquakes and tsunamis

TIDs create oscillations in electron density that affect HF propagation, causing:
- Rapid signal fading
- Doppler shifts
- Path distortion

**Detection approach:** Look for elevated SNR variance compared to baseline conditions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timezone

from ionis_jupyter import (
    load_dataset,
    grid_distance,
    solar_elevation,
    classify_path,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

## 2. Load Data

For TID detection, we need datasets with SNR variance (`snr_std` column).

In [ ]:
# Load WSPR signatures (best for TID work - calibrated SNR)
# If you don't have WSPR, contest or pskr also work
try:
    df = load_dataset("wspr", limit=1_000_000)  # Limit for demo
    source = "WSPR"
except FileNotFoundError:
    df = load_dataset("contest")
    source = "Contest"

print(f"Loaded {len(df):,} {source} signatures")
print(f"Columns: {list(df.columns)}")

## 3. Establish Baseline Variance

TID detection requires comparing observed variance to "normal" conditions.
We establish baselines per (band, hour, month) bucket.

In [ ]:
# Calculate baseline statistics per bucket
baseline = df.groupby(["band", "hour", "month"]).agg({
    "snr_std": ["mean", "std", "count"],
    "median_snr": "mean",
}).reset_index()

# Flatten column names
baseline.columns = ["_".join(col).strip("_") for col in baseline.columns]

print(f"Baseline buckets: {len(baseline):,}")
baseline.head()

In [ ]:
# Visualize baseline variance by band
fig, ax = plt.subplots(figsize=(12, 6))

BAND_NAMES = {
    102: "160m", 103: "80m", 104: "60m", 105: "40m", 106: "30m",
    107: "20m", 108: "17m", 109: "15m", 110: "12m", 111: "10m",
}

band_variance = baseline.groupby("band")["snr_std_mean"].mean()
bands = [BAND_NAMES.get(b, str(b)) for b in band_variance.index]

ax.bar(bands, band_variance.values, color="steelblue", alpha=0.7)
ax.set_xlabel("Band")
ax.set_ylabel("Mean SNR Standard Deviation (dB)")
ax.set_title("Baseline SNR Variance by Band")
plt.tight_layout()
plt.show()

## 4. Identify Anomalous Variance

Signatures with `snr_std` significantly above baseline may indicate TID activity.

In [ ]:
# Merge baseline stats with individual signatures
df_analysis = df.merge(
    baseline[["band", "hour", "month", "snr_std_mean", "snr_std_std"]],
    on=["band", "hour", "month"],
    how="left"
)

# Calculate z-score for variance
df_analysis["variance_zscore"] = (
    (df_analysis["snr_std"] - df_analysis["snr_std_mean"]) / 
    df_analysis["snr_std_std"].clip(lower=0.1)
)

# Flag anomalies (z > 2 = unusual variance)
df_analysis["is_anomaly"] = df_analysis["variance_zscore"] > 2

anomaly_rate = df_analysis["is_anomaly"].mean() * 100
print(f"Anomaly rate: {anomaly_rate:.2f}%")
print(f"Anomalous signatures: {df_analysis['is_anomaly'].sum():,}")

In [ ]:
# Distribution of variance z-scores
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(df_analysis["variance_zscore"].dropna(), bins=100, alpha=0.7, edgecolor="black")
ax.axvline(2, color="red", linestyle="--", label="Anomaly threshold (z=2)")
ax.axvline(0, color="gray", linestyle="-", alpha=0.5)
ax.set_xlabel("Variance Z-Score")
ax.set_ylabel("Count")
ax.set_title("Distribution of SNR Variance Relative to Baseline")
ax.set_xlim(-5, 10)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Spatial Analysis

TIDs propagate spatially. Anomalies should cluster geographically.

In [ ]:
# Anomaly rate by TX grid
grid_anomalies = df_analysis.groupby("tx_grid_4").agg({
    "is_anomaly": "mean",
    "variance_zscore": "mean",
    "median_snr": "count"
}).rename(columns={"median_snr": "sig_count"})

# Filter to grids with enough samples
grid_anomalies = grid_anomalies[grid_anomalies["sig_count"] >= 100]

# Top anomaly grids
print("Grids with highest anomaly rates:")
print(grid_anomalies.sort_values("is_anomaly", ascending=False).head(20))

## 6. Time-Series Analysis

For event-based TID detection (e.g., rocket launches), analyze variance over time.

In [ ]:
# Aggregate variance by hour (across all paths)
hourly_variance = df_analysis.groupby("hour").agg({
    "snr_std": "mean",
    "variance_zscore": "mean",
    "is_anomaly": "mean",
}).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Mean variance by hour
axes[0].bar(hourly_variance["hour"], hourly_variance["snr_std"], 
            color="steelblue", alpha=0.7)
axes[0].set_xlabel("Hour (UTC)")
axes[0].set_ylabel("Mean SNR Std Dev (dB)")
axes[0].set_title("SNR Variance by Hour")
axes[0].set_xticks(range(0, 24, 2))

# Right: Anomaly rate by hour
axes[1].bar(hourly_variance["hour"], hourly_variance["is_anomaly"] * 100,
            color="coral", alpha=0.7)
axes[1].set_xlabel("Hour (UTC)")
axes[1].set_ylabel("Anomaly Rate (%)")
axes[1].set_title("Anomaly Rate by Hour")
axes[1].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.show()

## 7. Path-Specific Analysis

For TID research, focus on specific paths that cross the region of interest.

In [ ]:
# Example: Find paths crossing a specific region
# Adjust these grids for your study area

def paths_crossing_region(df, region_prefixes):
    """Find paths where TX or RX is in the region."""
    tx_match = df["tx_grid_4"].str[:2].isin(region_prefixes)
    rx_match = df["rx_grid_4"].str[:2].isin(region_prefixes)
    return df[tx_match | rx_match]

# Example: UK region (IO, JO fields)
uk_paths = paths_crossing_region(df_analysis, ["IO", "JO"])
print(f"Paths crossing UK region: {len(uk_paths):,}")

# Anomaly rate for UK paths
if len(uk_paths) > 0:
    uk_anomaly_rate = uk_paths["is_anomaly"].mean() * 100
    print(f"UK region anomaly rate: {uk_anomaly_rate:.2f}%")

## 8. Recommendations for TID Research

### Data Selection
- **WSPR** is best for TID work — calibrated SNR, consistent 2-minute cycles
- Focus on **mid-band frequencies** (20m, 30m, 40m) for F-layer TIDs
- Use **single-hop paths** (< 3000 km) for cleaner signatures

### Baseline Building
- Build baselines from quiet-time data (low Kp, no known events)
- Account for seasonal and diurnal variation
- Use at least 30 days of data for robust baselines

### Event Analysis
- For rocket launches: focus on 1-2 hours post-launch
- For geomagnetic storms: look for onset signatures
- For eclipses: track the path of totality

### Related Resources
- W2NAF's 2022 GRL paper on large-scale TIDs in WSPR/RBN
- WSPRDaemon enhanced parameters (Doppler, drift)
- SuperDARN and GNSS TEC for complementary data

## 9. Case Study: SpaceX Launch TID — November 17, 2025

On November 17, 2025 at 05:21 UTC, a SpaceX Falcon 9 carrying Sentinel-6B launched from
Vandenberg SFB (SLC-4E). Researcher Isobel Smith (HamSCI mailing list) observed a **2x
increase in WSPR 20m spots** (39,989 vs ~20,000 on surrounding days) during the 05-07z
window — consistent with a rocket-induced TID enhancing F-layer propagation.

**The critical question:** Was the spot increase caused by a TID, or by changing
solar/geomagnetic conditions?

A companion dataset (`data/vandenberg-tid/`) extends this analysis to **12 Vandenberg
launches** (2022-2025) with verified dates, control windows, and solar conditions. Three
of twelve show statistically significant spot increases (z > 2): Sentinel-6B (z=6.2),
NROL-87 (z=2.87), and SPHEREx/PUNCH (z=2.13).

### Solar Context

We pulled 3-hour resolution solar indices from `solar.bronze` (GFZ Potsdam + NOAA SWPC)
to check for confounds on the Sentinel-6B launch.

In [ ]:
# Solar conditions Nov 15-19, 2025 (3-hour resolution)
# Source: GFZ Potsdam (SFI/SSN daily), NOAA SWPC (Kp 3-hour)

solar_data = {
    "date": ["2025-11-15"] * 8 + ["2025-11-16"] * 8 + ["2025-11-17"] * 8 +
            ["2025-11-18"] * 8 + ["2025-11-19"] * 8,
    "utc_window": ["00-03z", "03-06z", "06-09z", "09-12z",
                   "12-15z", "15-18z", "18-21z", "21-00z"] * 5,
    "sfi": [132.3]*8 + [132.3]*8 + [122.3]*8 + [120.1]*8 + [122.5]*8,
    "ssn": [100]*8 + [71]*8 + [57]*8 + [62]*8 + [54]*8,
    "kp": [1.00, 1.00, 2.33, 2.33, 3.33, 3.33, 1.67, 1.67,   # Nov 15
           2.67, 2.67, 3.67, 3.67, 3.00, 3.00, 2.67, 2.67,   # Nov 16
           2.67, 2.67, 3.00, 3.00, 3.67, 3.67, 1.00, 1.00,   # Nov 17
           2.33, 2.33, 1.33, 1.33, 0.00, 0.00, 0.67, 0.67,   # Nov 18
           0.67, 0.67, 1.00, 1.00, 0.33, 0.33, 0.00, 0.00],  # Nov 19
}

solar_df = pd.DataFrame(solar_data)

# Daily summary
daily = solar_df.groupby("date").agg(
    sfi=("sfi", "first"),
    ssn=("ssn", "first"),
    kp_min=("kp", "min"),
    kp_max=("kp", "max"),
    kp_mean=("kp", "mean"),
).reset_index()

daily["date"] = pd.to_datetime(daily["date"])
daily

In [ ]:
# Visualize the solar confound analysis
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

launch_date = pd.Timestamp("2025-11-17")

# SFI trend
axes[0].plot(daily["date"], daily["sfi"], "o-", color="darkorange", linewidth=2)
axes[0].axvline(launch_date, color="red", linestyle="--", alpha=0.7, label="Launch")
axes[0].set_ylabel("Solar Flux Index (SFU)")
axes[0].set_title("SFI — Declining on Launch Day")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=45)

# SSN trend
axes[1].plot(daily["date"], daily["ssn"], "s-", color="steelblue", linewidth=2)
axes[1].axvline(launch_date, color="red", linestyle="--", alpha=0.7, label="Launch")
axes[1].set_ylabel("Sunspot Number")
axes[1].set_title("SSN — Nearly Halved by Launch Day")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=45)

# Kp range
axes[2].fill_between(daily["date"], daily["kp_min"], daily["kp_max"],
                     alpha=0.3, color="coral", label="Kp range")
axes[2].plot(daily["date"], daily["kp_mean"], "d-", color="firebrick", linewidth=2,
             label="Kp mean")
axes[2].axvline(launch_date, color="red", linestyle="--", alpha=0.7, label="Launch")
axes[2].set_ylabel("Kp Index")
axes[2].set_title("Kp — Nov 16 Was Worse")
axes[2].legend(fontsize=8)
axes[2].tick_params(axis="x", rotation=45)

fig.suptitle("Solar Conditions: Nov 15-19, 2025 — SpaceX Launch TID Case Study",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Key Finding: Solar Decline Strengthens the TID Argument

| Metric | Nov 15 | Nov 17 (Launch) | Change |
|--------|--------|-----------------|--------|
| SFI | 132.3 | 122.3 | -10.0 SFU (-7.6%) |
| SSN | 100 | 57 | -43 (-43%) |
| Kp (05-07z) | 1.0 | 2.7-3.0 | Mildly unsettled |
| WSPR 20m spots | ~20,000 | 39,989 | +100% |

**The solar flux and sunspot number both declined** from Nov 15 to Nov 17. Under normal
ionospheric conditions, this should *reduce* 20m F-layer propagation — fewer spots, lower SNR.

Yet Smith observed a **2x increase** in WSPR spots on launch day.

This is a natural control: the background ionosphere was getting *worse* for propagation,
so the observed enhancement must have a non-solar cause. The TID hypothesis is consistent
with this — rocket exhaust creates gravity waves that modulate F-layer electron density,
temporarily enhancing (or disrupting) specific propagation paths.

**On Kp:** Nov 17 at 05-07z shows Kp 2.7-3.0 (mildly unsettled), but Nov 16 peaked at
Kp 3.67 without a corresponding spot increase. Geomagnetic activity does not explain the
anomaly.

### Suggestions for Further Analysis

- **Multi-transmitter check**: Confirm the spot increase came from multiple independent
  TX stations, ruling out single-beacon equipment changes
- **SNR distribution**: A TID should shift the SNR distribution (bimodal or broadened),
  not just increase spot count at the same SNR levels
- **Path directionality**: TIDs propagate with a wavefront — paths perpendicular to the
  wavefront should show different signatures than paths parallel to it
- **Time resolution**: WSPR's 2-minute cycle can resolve TID periods of 10-60 minutes
  (medium-scale TIDs) — look for periodic SNR oscillations in individual paths